In [1]:
from torch.backends.cudnn import benchmark

from data.iris_benchmarks.benchmarks.helpers import load_seed_points
from pydrake.all import (
    StartMeshcat,
    AddDefaultVisualization,
    Simulator,
    RobotDiagramBuilder,
    VPolytope,
    HPolyhedron,
    SceneGraphCollisionChecker,
    RandomGenerator,
    PointCloud,
    Rgba,
    Quaternion,
    RigidTransform,
    IrisFromCliqueCoverOptions,
    IrisInConfigurationSpaceFromCliqueCoverV2,
    SaveIrisRegionsYamlFile,
    LoadModelDirectives,
    ProcessModelDirectives,
    Sphere
)
import numpy as np
import importlib
import os
from IPython import get_ipython, extract_module_locals
import data.iris_benchmarks.benchmarks.helpers as benchmark_helpers 
from data.iris_benchmarks.iris_environments.environments import get_environment_builder

In [2]:
TEST_SCENE = "7DOFIIWA"

src_directory = os.path.abspath(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
parent_directory = os.path.dirname(src_directory)
data_directory = os.path.join(parent_directory, "data")

region_file = os.path.join(data_directory, "iris_regions" + TEST_SCENE + ".yaml")

plant, scene_graph, diagram, diagram_context, plant_context, models, meshcat =  get_environment_builder(TEST_SCENE)(True)
diagram.ForcedPublish(diagram_context)
print("Model Names:")
for i, model in enumerate(models):
    print(f"models[{i}]: {model.model_name}")

INFO:drake:Meshcat listening for connections at http://localhost:7000


Model Names:
models[0]: iiwa
models[1]: wsg
models[2]: shelves1
models[3]: shelves2
models[4]: ground


In [8]:
seed_points = benchmark_helpers.load_seed_points(TEST_SCENE)
def view_points_as_spheres(points, meshcat, name, radius=0.01, color = Rgba(1,0,0,1)):
    for i, point in enumerate(points):
        cur_name = f"{name}_{i}"
        meshcat.SetObject(cur_name, Sphere(radius), color)
        meshcat.SetTransform(cur_name, RigidTransform(p=point[:, np.newaxis]))

end_effector_model_info = models[1]
print("End Effector Bodies:")
for body_index in plant.GetBodyIndices(end_effector_model_info.model_instance):
    print(plant.get_body(body_index).name())
end_effector = plant.GetBodyByName("body", end_effector_model_info.model_instance)
end_effector_frame_id = plant.GetBodyFrameIdOrThrow(end_effector.index())
def get_end_effector_pose(q, context = None):
    if context is None:
        context = plant.CreateDefaultContext()
    plant.SetPositions(context, q)
    return plant.EvalBodyPoseInWorld(context, end_effector).translation()

def plot_end_effector_configurations(configs, meshcat, name, radius=0.05, color = Rgba(1,0,0,1)):
    context = plant.CreateDefaultContext()
    task_space_points = []
    for q in configs:
        task_space_points.append(get_end_effector_pose(q, context))
    view_points_as_spheres(task_space_points, meshcat, name, radius, color)

plot_end_effector_configurations(seed_points, meshcat, "seed_points")

End Effector Bodies:
body
left_finger
right_finger


In [13]:
import time
for seed_point in seed_points:
    plant.SetPositions(plant_context, seed_point)
    diagram.ForcedPublish(diagram_context)
    # time.sleep(0.5)

shelf_seed_point_indices = list(range(1,6))
shelf_seed_points = seed_points[shelf_seed_point_indices]
    

In [46]:
end_effector_frame_id

<FrameId value=1545>